# 第 12 章:对齐 I —— DPO:从偏好学习

第 10 章我们用 LoRA 做了参数高效微调,第 11 章把推理工程跑通。至此模型已经能「对话」了 —— 但它说话**不一定讨人喜欢**。

SFT 教会模型「什么输入配什么输出」,但它学的是**统计分布**,不是**人类偏好**。模型可能生成语法正确但内容有害、啰嗦、或者答非所问的回答。

本章介绍 **DPO(Direct Preference Optimization)** —— 直接从偏好对(chosen / rejected)中学习,**不需要训练 reward model**,一步到位地把模型「对齐」到人类偏好。

> 本章的代码全部对应 `trainer/train_dpo.py`(228 行)。所有行号基于 minimind 源码。

## 12.0 本章路线图

DPO 的核心可以浓缩成一句话:**把偏好学习变成一个分类问题**。

| 节 | 概念 | 对应源码 |
|---|---|---|
| 12.1 | 为什么 SFT 后还要对齐 | — |
| 12.2 | DPO 核心思想:从 RLHF 到闭式解 | — |
| 12.3 | DPO 损失函数 | `train_dpo.py:~34-50 (@67f114a)` |
| 12.4 | 参考模型(ref model) | `train_dpo.py:~188-193 (@67f114a)` |
| 12.5 | DPODataset 数据格式 | `lm_dataset.py:122-192` |
| 12.6 | logits → log_probs | `train_dpo.py:~25-31 (@67f114a)` |
| 12.7 | 训练配置(lr=4e-8) | `train_dpo.py:~135-154 (@67f114a)` |
| 12.8 | 效果对比:SFT vs DPO | — |

> 一句话总结:**DPO = 让模型在 chosen 和 rejected 之间「选对边」,同时别离 SFT 模型太远。**

&nbsp;

---

## Part 1:为什么需要对齐

### 12.1 为什么 SFT 后还要对齐

SFT(Supervised Fine-Tuning)的损失是 next-token prediction:

$$\mathcal{L}_{\text{SFT}} = -\sum_t \log \pi_\theta(y_t \mid y_{<t}, x)$$

它教模型「模仿训练数据」。但训练数据是互联网上爬来的,质量参差不齐。SFT 之后,模型可能:

- **不安全**:生成有害、歧视性内容
- **低质量**:答非所问、啰嗦、编造事实(hallucination)
- **不对齐**:不遵循用户的真实意图(比如用户问「帮我写代码」,模型却长篇大论讲概念)

解决这个问题的经典路线是 **RLHF(Reinforcement Learning from Human Feedback)**:

```
SFT → 训练 reward model → 用 PPO 优化策略 → 对齐
```

但 RLHF 极其复杂:需要训练一个独立的 reward model,需要 PPO 的 actor-critic 框架,需要处理 KL 散度约束,训练极不稳定。

**DPO 的洞察**:RLHF 的最优策略有**闭式解**(closed-form solution),可以把 reward model 直接消去,变成一个**纯监督的损失函数**。一步到位,不需要 reward model,不需要 RL。

### 12.2 DPO 的数学起源:从偏好到损失

DPO 的推导起点是 **Bradley-Terry 模型** —— 人类偏好可以用 reward 的差来表达:

$$P(y_w \succ y_l \mid x) = \sigma\!\left(r(x, y_w) - r(x, y_l)\right)$$

其中 $y_w$ 是 chosen(win),$y_l$ 是 rejected(loss),$\sigma$ 是 sigmoid。

RLHF 的目标是最大化 reward,同时用 KL 散度约束策略不要偏离参考模型 $\pi_{\text{ref}}$ 太远:

$$\max_\pi \; \mathbb{E}[r(x,y)] - \beta \, \text{KL}(\pi \| \pi_{\text{ref}})$$

这个优化问题有**闭式解**:

$$\pi^*(y \mid x) = \frac{1}{Z(x)} \pi_{\text{ref}}(y \mid x) \exp\!\left(\frac{1}{\beta} r(x, y)\right)$$

> 注意 $Z(x)$ 是配分函数(partition function),只依赖 $x$,不依赖 $y$。这个细节是 DPO 能消去 reward model 的关键。

从闭式解中把 reward $r(x,y)$ **反解**出来:

$$r(x, y) = \beta \log \frac{\pi^*(y \mid x)}{\pi_{\text{ref}}(y \mid x)} + \beta \log Z(x)$$

现在把这个 reward 代回 Bradley-Terry 偏好模型 $P(y_w \succ y_l) = \sigma(r(x,y_w) - r(x,y_l))$。奇妙的是:$Z(x)$ 项在 $y_w - y_l$ 的差中**完全消掉了**!最终得到 DPO loss:

> **这就是 DPO 的全部魔法:把 reward model 消掉,直接用策略本身的 log-prob 来表达偏好。**

&nbsp;

---

## Part 2:DPO 损失函数

### 12.3 DPO 损失函数

DPO 的损失函数(`train_dpo.py:~34-50 (@67f114a)`):

$$\boxed{\mathcal{L}_{\text{DPO}} = -\log \sigma\!\left(\beta \left[\big(\log \pi(y_w) - \log \pi(y_l)\big) - \big(\log \pi_{\text{ref}}(y_w) - \log \pi_{\text{ref}}(y_l)\big)\right]\right)}$$

拆开看这个公式的每一项:

| 项 | 含义 |
|---|---|
| $\log \pi(y_w) - \log \pi(y_l)$ | 当前策略对 chosen vs rejected 的**对数概率差** |
| $\log \pi_{\text{ref}}(y_w) - \log \pi_{\text{ref}}(y_l)$ | 参考模型的对应差(基线) |
| 两者相减 | 策略相对于参考模型,在偏好上的**提升量** |
| $\beta$ | 控制约束强度(温度) |
| $-\log\sigma(\cdot)$ | 让提升量为正时 loss → 0,为负时 loss → $\infty$ |

直觉:**DPO 鼓励策略比参考模型「更偏好 chosen」**。如果策略已经比 ref 更偏 chosen,loss 就小;如果策略反而更偏 rejected,loss 就大。

下面先用 naive 写法手算这个 loss,再看 minimind 的紧凑实现。

In [ ]:
import torch
import torch.nn.functional as F

# === naive DPO loss:完全展开,对应 train_dpo.py:~34-50 (@67f114a) ===

def dpo_loss_naive(chosen_logp_policy, rejected_logp_policy,
                   chosen_logp_ref, rejected_logp_ref, beta=0.15):
    """
    chosen_logp_policy:  策略模型对 chosen 序列的 log-prob 之和  shape: (B/2,)
    rejected_logp_policy: 策略模型对 rejected 序列的 log-prob 之和
    chosen_logp_ref:     参考模型对 chosen 序列的 log-prob 之和
    rejected_logp_ref:   参考模型对 rejected 序列的 log-prob 之和
    """
    # Step 1: 策略的对数概率差(chosen vs rejected)
    pi_logratios = chosen_logp_policy - rejected_logp_policy

    # Step 2: 参考模型的对数概率差(基线)
    ref_logratios = chosen_logp_ref - rejected_logp_ref

    # Step 3: 策略相对于参考模型的「偏好提升」
    logits = pi_logratios - ref_logratios

    # Step 4: DPO loss = -log(sigmoid(beta * logits))
    loss = -F.logsigmoid(beta * logits)
    return loss.mean()

# 模拟一个 batch:3 条偏好对
# 假设策略模型已经比 ref 更偏 chosen(提升量为正)
chosen_logp_policy  = torch.tensor([-5.0, -4.0, -6.0])
rejected_logp_policy = torch.tensor([-7.0, -6.0, -9.0])
chosen_logp_ref      = torch.tensor([-5.0, -4.0, -6.0])  # ref 和初始策略一样
rejected_logp_ref    = torch.tensor([-6.0, -5.0, -7.0])

loss = dpo_loss_naive(chosen_logp_policy, rejected_logp_policy,
                       chosen_logp_ref, rejected_logp_ref, beta=0.15)
print(f"DPO loss (naive): {loss.item():.4f}")

# 内部 logits:策略偏好提升量
pi_lr = chosen_logp_policy - rejected_logp_policy     # [2, 2, 3]
ref_lr = chosen_logp_ref - rejected_logp_ref           # [1, 1, 1]
logits = pi_lr - ref_lr                                # [1, 1, 2]
print(f"偏好提升 logits: {logits.tolist()}")
print(f"sigmoid(beta*logits): {torch.sigmoid(0.15 * logits).tolist()}")
# logits 全正 → sigmoid > 0.5 → loss 较小(策略方向正确)

> **$-\log\sigma(x)$ 的直觉**:当 $x > 0$(提升量为正)时,$\sigma(x) > 0.5$,$-\log\sigma \to 0$,loss 小。当 $x < 0$(策略反而更偏 rejected)时,$\sigma(x) < 0.5$,$-\log\sigma \to \infty$,loss 爆炸。这就是「分类损失」的本质 —— 把「选对边」变成一个二分类问题。

现在看 minimind 的紧凑实现 —— 它把 chosen 和 rejected 拼在同一个 batch 里,通过切分前后半段来区分:

In [ ]:
# === compact DPO loss:minimind 的实现 (train_dpo.py:~34-50 (@67f114a)) ===
# 关键技巧:chosen 和 rejected 被拼接在同一个 batch 中
# 前半 batch 是 chosen,后半是 rejected

def dpo_loss(ref_log_probs, policy_log_probs, mask, beta):
    """
    ref_log_probs:    (batch, seq_len) — 参考模型每个 token 的 log-prob
    policy_log_probs: (batch, seq_len) — 策略模型每个 token 的 log-prob
    mask:             (batch, seq_len) — 只算 assistant span 的 token
    batch = 2 * n_pairs(chosen 在前,rejected 在后)
    """
    # Step 1: 用 mask 把 padding 和 user span 清零,然后对 seq 求和
    ref_log_probs = (ref_log_probs * mask).sum(dim=1)         # (batch, seq_len) -> (batch,)
    policy_log_probs = (policy_log_probs * mask).sum(dim=1)   # (batch, seq_len) -> (batch,)

    # Step 2: 切分前后半段
    batch_size = ref_log_probs.shape[0]
    half = batch_size // 2
    chosen_ref   = ref_log_probs[:half]       # chosen 的 ref log-prob 之和
    rejected_ref = ref_log_probs[half:]       # rejected 的 ref log-prob 之和
    chosen_pol   = policy_log_probs[:half]    # chosen 的 policy log-prob 之和
    rejected_pol = policy_log_probs[half:]    # rejected 的 policy log-prob 之和

    # Step 3: 策略 log-ratio 减去参考 log-ratio
    pi_logratios  = chosen_pol - rejected_pol
    ref_logratios = chosen_ref - rejected_ref
    logits = pi_logratios - ref_logratios

    # Step 4: DPO loss
    loss = -F.logsigmoid(beta * logits)
    return loss.mean()

# 验证:构造 4 条(batch=2 对),前 2 条是 chosen,后 2 条是 rejected
torch.manual_seed(42)
B, T = 4, 6  # 2 pairs
ref_lp    = torch.randn(B, T) * 0.5 - 3   # 模拟 ref 的 per-token log-prob
policy_lp = ref_lp + torch.randn(B, T) * 0.2  # policy 略有偏移
mask = torch.ones(B, T)  # 简化:全部计入

loss_compact = dpo_loss(ref_lp, policy_lp, mask, beta=0.15)
print(f"DPO loss (compact): {loss_compact.item():.4f}")

# 验证两种写法等价(对同一组数据)
half = B // 2
naive = dpo_loss_naive(
    (policy_lp[:half] * mask[:half]).sum(1),
    (policy_lp[half:] * mask[half:]).sum(1),
    (ref_lp[:half] * mask[:half]).sum(1),
    (ref_lp[half:] * mask[half:]).sum(1),
    beta=0.15
)
print(f"DPO loss (naive,   对齐数据): {naive.item():.4f}")
print(f"两者差异: {abs(loss_compact.item() - naive.item()):.2e}")

### 12.4 参考模型(Reference Model)

DPO 公式里反复出现 $\pi_{\text{ref}}$ —— 参考模型。它的角色:**锚点**。

如果没有 ref model,DPO loss 退化成:

$$\mathcal{L} = -\log\sigma(\beta(\log\pi(y_w) - \log\pi(y_l)))$$

这会鼓励模型**无限抬高 chosen 的概率、无限压低 rejected 的概率**。结果:模型为了「选对边」,可能把所有概率质量都堆到几个 token 上,生成质量崩塌(模式坍塌 / mode collapse)。

ref model 的作用是:**你只能比 SFT 模型偏离一点点**。损失函数里减掉 ref 的 log-ratio,就是要求「在保持原有能力的前提下,往偏好方向微调」。

minimind 的实现(`train_dpo.py:~188-193 (@67f114a)`):

```python
# 加载策略模型(要训练的)
model, tokenizer = init_model(lm_config, args.from_weight, device=args.device)
# 加载参考模型(同一个 SFT 权重的副本,冻结)
ref_model, _ = init_model(lm_config, args.from_weight, device=args.device)
ref_model.eval()                  # 切到推理模式(关 dropout)
ref_model.requires_grad_(False)   # 冻结所有参数,不计算梯度
```

注意三点:
1. **同一个权重**:`from_weight='full_sft'`,ref 和 policy 初始完全相同
2. **`requires_grad_(False)`**:ref 不需要梯度,省显存
3. **`torch.no_grad()` 上下文**:训练循环里(`train_dpo.py:~74 (@67f114a)`),ref 的 forward 在 `no_grad` 下执行

> ref model 占用的显存和 policy 一样大。所以 DPO 训练的显存大约是 SFT 的 **2 倍**(两个模型 + optimizer state)。

In [ ]:
# 演示 ref model 的冻结效果

import torch.nn as nn

# 模拟一个极简模型
class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4, 4)

model = TinyModel()
ref_model = TinyModel()

# 复制权重(等价于 init_model 同一权重)
ref_model.load_state_dict(model.state_dict())

# 冻结 ref model
ref_model.eval()
ref_model.requires_grad_(False)

# 验证:ref model 的参数不参与梯度计算
x = torch.randn(2, 4)
out_policy = model(x)
out_ref = ref_model(x)
loss = (out_policy - out_ref).pow(2).mean()
loss.backward()

print("policy linear.weight grad is None?", model.linear.weight.grad is None)        # False
print("ref    linear.weight grad is None?", ref_model.linear.weight.grad)             # None(冻结了!)
print(f"ref model requires_grad: {ref_model.linear.weight.requires_grad}")           # False

# 关键:即使 ref_model 参与了 loss 计算,它的梯度也不会被计算
# 因为 requires_grad_(False) 切断了计算图

&nbsp;

---

## Part 3:数据与训练

### 12.5 DPODataset:偏好对的数据格式

DPO 的训练数据是**偏好对(preference pairs)**。每条数据的格式(`lm_dataset.py:122-174`):

```json
{
  "chosen": [
    {"role": "user", "content": "什么是 DPO?"},
    {"role": "assistant", "content": "DPO 是直接偏好优化..."}
  ],
  "rejected": [
    {"role": "user", "content": "什么是 DPO?"},
    {"role": "assistant", "content": "DPO 是一种数据库..."}
  ]
}
```

`chosen` 和 `rejected` 共享**相同的 prompt(用户输入)**,只有 assistant 的回答不同 —— 一个好(chosen),一个差(rejected)。

DPODataset 对每条数据做四件事:

1. 用 `apply_chat_template` 把对话渲染成文本
2. tokenize 成 `input_ids`(padding 到 `max_length`)
3. 用 `generate_loss_mask` 标记 assistant span 的 token(只有回答部分参与 loss)
4. 构造 `x`(input_ids[:-1])、`y`(input_ids[1:])、`mask`(loss_mask[1:])

`generate_loss_mask` 的逻辑(`lm_dataset.py:176-192`):扫描 `input_ids`,找到 `<bos>assistant\n` 标记的开始位置,到 `<eos>\n` 标记的结束位置,中间这段标记为 1,其余为 0。

In [ ]:
# 演示 generate_loss_mask 的逻辑

# 模拟 tokenized 序列(简化版)
# 假设 bos_id = [10, 20](代表 "<bos>assistant\n")
# 假设 eos_id = [30]    (代表 "<eos>\n")

bos_id = [10, 20]
eos_id = [30]

# 构造一条序列: [user_tokens... <bos>assistant\n assistant_content <eos>\n padding...]
input_ids = (
    [1, 2, 3]           # user prompt 部分(mask=0)
    + bos_id             # 10, 20 (mask=0,标记本身不计入)
    + [40, 41, 42, 43]   # assistant 回答内容(mask=1!)
    + eos_id             # 30 (mask=1,计入)
    + [0, 0, 0]          # padding(mask=0)
)
print(f"input_ids: {input_ids}")

# 简化版 generate_loss_mask
def generate_loss_mask(input_ids, bos_id, eos_id, max_length=4096):
    loss_mask = [0] * len(input_ids)
    i = 0
    while i < len(input_ids):
        if input_ids[i:i + len(bos_id)] == bos_id:
            start = i + len(bos_id)  # 跳过 bos 标记本身
            end = start
            # 找到 eos
            while end < len(input_ids):
                if input_ids[end:end + len(eos_id)] == eos_id:
                    break
                end += 1
            # 标记 [start, end + len(eos_id)) 为 1
            for j in range(start, min(end + len(eos_id), max_length)):
                loss_mask[j] = 1
            i = end + len(eos_id) if end < len(input_ids) else len(input_ids)
        else:
            i += 1
    return loss_mask

mask = generate_loss_mask(input_ids, bos_id, eos_id)
print(f"loss_mask: {mask}")
print(f"token  -> mask")
for idx, (tok, m) in enumerate(zip(input_ids, mask)):
    label = "assistant span" if m == 1 else "(ignored)"
    print(f"  [{idx}] tok={tok:>3}  mask={m}  {label}")

# DPO loss 只在 mask=1 的位置计算 log-prob
# 这意味着:模型只在「assistant 的回答部分」上学习偏好
# prompt 和 padding 不影响梯度

> **为什么只标 assistant span?** 因为 DPO 优化的是「**给定相同 prompt,生成 chosen 还是 rejected**」。prompt 对 chosen 和 rejected 是一样的,在 prompt 上算 loss 没有意义。只有回答部分不同,才携带偏好信号。

### 12.6 logits_to_log_probs:从 logits 到序列概率

DPO loss 需要 $\log \pi(y)$ —— 模型给整个回答序列分配的对数概率。这需要两步:

1. 模型输出 logits:`(batch, seq_len, vocab_size)`
2. 把 logits 转成每个 token 的 log-prob:`(batch, seq_len)`

minimind 的实现(`train_dpo.py:~25-31 (@67f114a)`):

```python
def logits_to_log_probs(logits, labels):
    log_probs = F.log_softmax(logits, dim=2)           # (B, T, V)
    log_probs_per_token = torch.gather(
        log_probs, dim=2, index=labels.unsqueeze(2)    # 取出 label 对应的 log-prob
    ).squeeze(-1)                                      # (B, T, V) -> (B, T)
    return log_probs_per_token
```

三个关键操作:

| 操作 | 作用 | Shape |
|---|---|---|
| `log_softmax(dim=2)` | 把 logits 转成对数概率(每行和为 0 对应概率和为 1) | (B,T,V) → (B,T,V) |
| `gather(dim=2, index=labels)` | 只取出「正确 token」对应的 log-prob | (B,T,V) → (B,T,1) |
| `squeeze(-1)` | 去掉多余维度 | (B,T,1) → (B,T) |

> 注意:这里得到的是**每个 token** 的 log-prob。序列的 log-prob 是所有 token 的 log-prob **之和**(在 `dpo_loss` 里用 `(lp * mask).sum(dim=1)` 完成)。

In [ ]:
# 演示 logits_to_log_probs 的每一步

B, T, V = 2, 3, 5  # batch=2, seq_len=3, vocab=5

torch.manual_seed(0)
logits = torch.randn(B, T, V)  # 模型输出的 logits
labels = torch.tensor([[1, 3, 0],
                       [2, 4, 1]])  # 正确的 token id

# Step 1: log_softmax —— 每个位置变成对数概率分布
log_probs = F.log_softmax(logits, dim=2)  # (B, T, V)
print(f"logits shape:    {logits.shape}")     # (2, 3, 5)
print(f"log_probs shape: {log_probs.shape}")  # (2, 3, 5)
print(f"log_probs[0,0,:] 和 = {log_probs[0,0,:].sum().item():.4f}")  # 不是 1!是负数(log)
print(f"exp(log_probs[0,0,:]) 和 = {log_probs[0,0,:].exp().sum().item():.4f}")  # = 1.0

# Step 2: gather —— 取出 label 对应位置的 log-prob
# labels.unsqueeze(2): (B, T) -> (B, T, 1),作为 gather 的 index
gathered = torch.gather(log_probs, dim=2, index=labels.unsqueeze(2))  # (B, T, V) -> (B, T, 1)
print(f"\ngathered shape: {gathered.shape}")  # (2, 3, 1)
print(f"gathered:\n{gathered.squeeze(-1)}")

# Step 3: squeeze
log_probs_per_token = gathered.squeeze(-1)  # (B, T, 1) -> (B, T)
print(f"\nper-token log-prob shape: {log_probs_per_token.shape}")  # (2, 3)

# 序列的 log-prob = 所有 token log-prob 之和
seq_log_prob = log_probs_per_token.sum(dim=1)  # (B, T) -> (B,)
print(f"\n序列 log-prob: {seq_log_prob.tolist()}")  # 每条序列一个值(负数,越接近 0 越好)

# 对比:手算第一条序列 token=1 的 log-prob
manual = log_probs[0, 0, 1].item()
print(f"\n手算验证: log_probs[0,0,label=1] = {manual:.4f}, gather 结果 = {log_probs_per_token[0,0].item():.4f}")

### 12.7 训练配置:lr=4e-8,极低!

DPO 的训练配置和 SFT 有一个**惊人的差异**:学习率。

| 配置 | SFT | DPO | 倍数 |
|---|---|---|---|
| `learning_rate` | 1e-5 | **4e-8** | SFT 的 1/250 |
| `epochs` | 3 | **1** | 1/3 |
| `batch_size` | 32 | 4 | 1/8 |
| `beta` | — | **0.15** | DPO 专属 |

为什么 lr 这么低?因为 DPO 是在 SFT 模型基础上做**微调**,目的是「轻轻推一把」,不是「重新学」。lr 太高会导致:

- **灾难性遗忘(catastrophic forgetting)**:模型忘了 SFT 学到的能力
- **策略崩塌**:模型为了降低 DPO loss,把概率分布推到极端,生成质量下降

`beta=0.15` 控制约束强度:

- $\beta$ **大**(如 1.0):约束弱,模型可以大幅偏离 ref → 过拟合风险
- $\beta$ **小**(如 0.01):约束强,模型几乎不动 → 学不到偏好

minimind 的默认配置(`train_dpo.py:~135-154 (@67f114a)`):

```python
parser.add_argument("--epochs", type=int, default=1)          # 只训 1 轮
parser.add_argument("--learning_rate", type=float, default=4e-8)  # 极低
parser.add_argument("--batch_size", type=int, default=4)      # 小 batch
parser.add_argument('--beta', default=0.15, type=float)        # DPO beta
parser.add_argument('--from_weight', default='full_sft')       # 基于 SFT 权重
```

> **🔧 后缀逻辑更新**: DPO checkpoint 保存路径同样走 `_model_suffix(lm_config)` (见 [ch08 §8.8](../ch08/01_main-chapter-code/ch08.ipynb)), 按 `_ple`/`_moe`/空 区分。**默认 Dense 模式后缀为空**, checkpoint 路径 (如 `dpo_768.pth`) 与上游原版一致。

In [ ]:
# 演示 lr 和 beta 对 DPO 的极端影响

import torch
import torch.nn.functional as F

def compute_dpo_loss(chosen_pol, rejected_pol, chosen_ref, rejected_ref, beta):
    pi_lr = chosen_pol - rejected_pol
    ref_lr = chosen_ref - rejected_ref
    return -F.logsigmoid(beta * (pi_lr - ref_lr)).mean()

# 初始状态:policy == ref(训练刚开始)
chosen_ref = torch.tensor([-5.0, -5.0, -5.0])
rejected_ref = torch.tensor([-6.0, -6.0, -6.0])
chosen_pol = chosen_ref.clone()
rejected_pol = rejected_ref.clone()

print("=== beta 对 DPO loss 的影响 ===")
print(f"{'beta':>6} | {'logits':>8} | {'sigmoid':>8} | {'loss':>8}")
print("-" * 42)
for beta in [0.01, 0.05, 0.15, 0.5, 1.0]:
    # 模拟训练后策略提升了 chosen 的概率
    pol_chosen = chosen_ref + 0.5  # chosen 概率上升
    pol_rejected = rejected_ref     # rejected 不变
    logits = (pol_chosen - pol_rejected) - (chosen_ref - rejected_ref)
    sig = torch.sigmoid(beta * logits).mean()
    loss = compute_dpo_loss(pol_chosen, pol_rejected, chosen_ref, rejected_ref, beta)
    print(f"{beta:>6.2f} | {logits.mean():>8.3f} | {sig:>8.4f} | {loss:>8.4f}")

print("\n=== lr 过高导致的问题模拟 ===")
# 如果 lr 很高,一个 epoch 就把 chosen logp 推到 -1,rejected 推到 -20
extreme_chosen = torch.tensor([-1.0, -1.0, -1.0])
extreme_rejected = torch.tensor([-20.0, -20.0, -20.0])
extreme_logits = (extreme_chosen - extreme_rejected) - (chosen_ref - rejected_ref)
print(f"极端 logits: {extreme_logits.mean():.1f} (模型为选对边把概率推到极端)")
print(f"极端 loss: {compute_dpo_loss(extreme_chosen, extreme_rejected, chosen_ref, rejected_ref, 0.15):.6f}")
print("loss 虽然很低,但模型生成质量已经崩塌(mode collapse)")

### 12.8 效果对比:SFT vs DPO

DPO 训练前后,模型对 chosen 和 rejected 的概率分配会发生变化。我们用一个 toy 实验模拟这个过程:

- **训练前**(policy == ref):$\log\pi(y_w) - \log\pi(y_l)$ 的差和 ref 一样,logits = 0
- **训练后**:策略应该让 chosen 概率升高、rejected 概率降低,logits > 0

训练循环的核心(`train_dpo.py:~57-95 (@67f114a)`):

```python
for step, batch in enumerate(loader):
    x = torch.cat([x_chosen, x_rejected], dim=0)     # 拼接 chosen + rejected
    y = torch.cat([y_chosen, y_rejected], dim=0)
    mask = torch.cat([mask_chosen, mask_rejected], dim=0)

    with torch.no_grad():
        ref_logits = ref_model(x).logits             # ref 不算梯度
    ref_log_probs = logits_to_log_probs(ref_logits, y)

    policy_log_probs = logits_to_log_probs(model(x).logits, y)  # policy 算梯度

    loss = dpo_loss(ref_log_probs, policy_log_probs, mask, beta) + aux_loss
    loss.backward()
    optimizer.step()
```

注意 batch 构造的巧妙之处:**chosen 和 rejected 拼在同一个 batch 里**,前半是 chosen,后半是 rejected。这样一个 forward 就算完两者,然后在 `dpo_loss` 里切分。

In [ ]:
# 模拟 DPO 训练前后的概率变化

import torch
import torch.nn.functional as F

torch.manual_seed(42)

# 模拟 3 条偏好对
n_pairs = 3

# 训练前:policy == ref
ref_chosen_lp   = torch.tensor([-8.2, -7.5, -9.1])
ref_rejected_lp = torch.tensor([-8.5, -7.8, -9.0])

# 训练后:policy 偏向 chosen(chosen 升,rejected 降)
post_chosen_lp   = torch.tensor([-7.1, -6.8, -8.2])   # 概率升高(logp 更接近 0)
post_rejected_lp = torch.tensor([-9.3, -8.9, -10.1])   # 概率降低(logp 更负)

def show_dpo_metrics(chosen_lp, rejected_lp, ref_chosen, ref_rejected, label, beta=0.15):
    """计算 DPO 相关指标"""
    # 策略的 chosen-rejected 差
    pol_margin = (chosen_lp - rejected_lp).mean().item()
    # ref 的 chosen-rejected 差
    ref_margin = (ref_chosen - ref_rejected).mean().item()
    # DPO logits(提升量)
    logits = ((chosen_lp - rejected_lp) - (ref_chosen - ref_rejected)).mean().item()
    # 隐式 reward
    reward = beta * (chosen_lp - ref_chosen).mean().item()
    # DPO loss
    pi_lr = chosen_lp - rejected_lp
    ref_lr = ref_chosen - ref_rejected
    loss = -F.logsigmoid(beta * (pi_lr - ref_lr)).mean().item()

    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    print(f"  策略 chosen-rejected margin: {pol_margin:+.3f}")
    print(f"  参考 chosen-rejected margin: {ref_margin:+.3f}")
    print(f"  DPO logits (提升量):        {logits:+.3f}")
    print(f"  隐式 reward (chosen):        {reward:+.4f}")
    print(f"  DPO loss:                    {loss:.4f}")
    # 准确率:logits > 0 表示策略方向正确
    acc = ((pi_lr - ref_lr) > 0).float().mean().item()
    print(f"  偏好准确率 (logits > 0):     {acc:.1%}")

# 训练前
show_dpo_metrics(ref_chosen_lp, ref_rejected_lp,
                 ref_chosen_lp, ref_rejected_lp, "训练前 (policy == ref)")
# 训练后
show_dpo_metrics(post_chosen_lp, post_rejected_lp,
                 ref_chosen_lp, ref_rejected_lp, "训练后 (policy 偏向 chosen)")

print(f"\n{'='*50}")
print(f"结论:DPO logits 从 0 提升到正值,loss 下降,偏好准确率上升。")
print(f"模型学会了在 chosen 和 rejected 之间「选对边」。")

## Summary and takeaways

本章我们从「为什么 SFT 不够」出发,完整拆解了 DPO 的原理和实现:

| 概念 | 要点 | 对应代码 |
|---|---|---|
| SFT 的局限 | 学统计分布,不学人类偏好 | — |
| RLHF → DPO | 消去 reward model,闭式解 | — |
| DPO loss | $-\log\sigma(\beta[(\Delta\log\pi) - (\Delta\log\pi_{\text{ref}})])$ | `train_dpo.py:~34-50 (@67f114a)` |
| 参考模型 | 冻结的 SFT 副本,防止策略偏离 | `train_dpo.py:~188-193 (@67f114a)` |
| 偏好数据 | `{"chosen":[...], "rejected":[...]}` | `lm_dataset.py:122-174` |
| loss mask | 只在 assistant span 上算 loss | `lm_dataset.py:176-192` |
| logits → logp | log_softmax → gather → sum | `train_dpo.py:~25-31 (@67f114a)` |
| 训练配置 | lr=4e-8(极低)、epochs=1、beta=0.15 | `train_dpo.py:~135-154 (@67f114a)` |

> **核心认知**:DPO 的优雅之处在于——它把 RLHF 中「训练 reward model → PPO 优化」的复杂 pipeline,压缩成了一个**可微分的分类损失**。模型直接从偏好对中学习,梯度信号清晰,训练稳定。代价是:需要显式维护一个冻结的 ref model(2 倍显存),且 lr 必须极低(防止灾难性遗忘)。

下一章我们将进入对齐的进阶内容,探讨更多偏好学习方法。

- 精简复习版见 [`./dpo.ipynb`](./dpo.ipynb)
- 本章习题与解答见 [`./exercise-solutions.ipynb`](./exercise-solutions.ipynb)